# Titans + MIRAS Memory-as-Context Promoter Classification

This tutorial is a **simplified educational SeqTrainer prototype** inspired by Titans/MIRAS concepts, not an official Google implementation.

## 1. Motivation

MIRAS design choices exposed in this prototype:
1. **Memory architecture**: MLP-based neural memory bank.
2. **Attentional bias / memory objective**: associative retrieval via query-key scores; surprise-proxy represented by embedding mismatch pressure through supervised loss.
3. **Retention gate / forgetting regularizer**: EMA-style retention gate for memory state updates.
4. **Memory algorithm / optimizer**: standard gradient descent (AdamW) for model params plus internal retention update for memory state.

Titans MAC approximation used here:
- short-term token encoder (Transformer over current sequence)
- long-term memory summarizing past chunks
- optional persistent task memory tokens
- Memory-as-Context by prepending retrieved memory vectors to the current sequence embeddings

## 2. Setup

Install torch extra if needed:
```bash
pip install -e '.[torch]'
```

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from seqtrainer.torch import DNATokenizer, TitansMIRASConfig, TitansMemoryAsContextClassifier

device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print('device:', device)

## 3. Load CSV data

In [ ]:
MAX_TRAIN_ROWS = 8000
MAX_EVAL_ROWS = 2000
MAX_TEST_ROWS = 2000

def resolve_csv(name):
    candidates = [Path('../../') / name, Path(name)]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f'Could not find {name} in {candidates}')

def load_split(filename, max_rows=None):
    path = resolve_csv(filename)
    df = pd.read_csv(path, usecols=['sequence', 'label'])
    if max_rows is not None:
        df = df.head(max_rows)
    return df

train_df = load_split('train_EP_DNA_BERT2_genomic_order.csv', MAX_TRAIN_ROWS)
eval_df = load_split('eval_EP_DNA_BERT2_genomic_order.csv', MAX_EVAL_ROWS)
test_df = load_split('test_EP_DNA_BERT2_genomic_order.csv', MAX_TEST_ROWS)

for name, df in [('train', train_df), ('eval', eval_df), ('test', test_df)]:
    lengths = df['sequence'].str.len()
    print(name, df.shape, 'label_mean=', round(df['label'].mean(), 4), 'len_mean=', round(lengths.mean(), 2), 'len_max=', int(lengths.max()))

## 4. Tokenization and Dataset

In [ ]:
class PromoterDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.seqs = df['sequence'].astype(str).tolist()
        self.labels = df['label'].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        ids, mask = self.tokenizer.encode(self.seqs[idx], max_length=self.max_length)
        return {'input_ids': torch.tensor(ids), 'attention_mask': torch.tensor(mask), 'labels': torch.tensor(self.labels[idx])}

max_length = 512
tokenizer = DNATokenizer(max_length=max_length)
train_ds = PromoterDataset(train_df, tokenizer, max_length=max_length)
eval_ds = PromoterDataset(eval_df, tokenizer, max_length=max_length)
test_ds = PromoterDataset(test_df, tokenizer, max_length=max_length)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
eval_loader = DataLoader(eval_ds, batch_size=64)
test_loader = DataLoader(test_ds, batch_size=64)

## 5. Model

In [ ]:
config = TitansMIRASConfig(max_length=max_length, d_model=128, num_heads=4, num_layers=2, memory_slots=8, memory_depth=2, memory_context_tokens=4, num_classes=2, memory_architecture='mlp', attentional_bias='mse_associative_surprise', retention_gate=0.9, memory_algorithm='adamw_plus_ema_memory_update', use_persistent_memory=True)
model = TitansMemoryAsContextClassifier(config).to(device)
print('params:', format(sum(p.numel() for p in model.parameters()), ','))
print(config.to_dict())

## 6. Training loop

In [ ]:
EPOCHS = 2
LR = 2e-4
MAX_TRAIN_STEPS_PER_EPOCH = 150
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

def run_epoch(loader, train=True):
    model.train(train)
    total_loss = 0.0
    correct = 0
    total = 0
    it = tqdm(loader, leave=False) if tqdm is not None else loader
    if not train:
        model.reset_memory()
    for step, batch in enumerate(it):
        if train and step >= MAX_TRAIN_STEPS_PER_EPOCH:
            break
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        with torch.set_grad_enabled(train):
            out = model(input_ids=ids, attention_mask=mask, labels=labels, update_memory=True)
            loss = out['loss']
            logits = out['logits']
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        total_loss += float(loss.item()) * labels.size(0)
        correct += int((logits.argmax(-1) == labels).sum().item())
        total += labels.size(0)
    return {'loss': total_loss / max(total, 1), 'acc': correct / max(total, 1)}

for epoch in range(1, EPOCHS + 1):
    print('epoch', epoch, 'train', run_epoch(train_loader, True), 'eval', run_epoch(eval_loader, False))

## 7. Evaluation

In [ ]:
def evaluate(loader, name):
    model.eval()
    y_true, y_pred = [], []
    model.reset_memory()
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            logits = model(ids, attention_mask=mask, update_memory=True)
            preds = logits.argmax(-1)
            y_true.extend(labels.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())
    y_true = np.array(y_true); y_pred = np.array(y_pred)
    print(name, 'accuracy', float((y_true == y_pred).mean()))
    try:
        from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
        print('precision', precision_score(y_true, y_pred, zero_division=0))
        print('recall', recall_score(y_true, y_pred, zero_division=0))
        print('f1', f1_score(y_true, y_pred, zero_division=0))
        print('confusion_matrix\n', confusion_matrix(y_true, y_pred))
    except Exception as exc:
        print('sklearn metrics unavailable:', exc)

evaluate(eval_loader, 'eval')
evaluate(test_loader, 'test')

## 8. Memory inspection

In [ ]:
batch = next(iter(eval_loader))
ids = batch['input_ids'].to(device)
mask = batch['attention_mask'].to(device)
model.reset_memory()
_ = model(ids, attention_mask=mask, update_memory=True)
state_after_update = model.long_term_memory.memory_state.detach().cpu().clone()
model.reset_memory()
state_after_reset = model.long_term_memory.memory_state.detach().cpu().clone()
with_memory = model(ids, attention_mask=mask, update_memory=True).argmax(-1).cpu()
model.reset_memory()
without_memory = model(ids, attention_mask=mask, update_memory=False).argmax(-1).cpu()
print('memory norm after update:', float(state_after_update.norm()))
print('memory norm after reset:', float(state_after_reset.norm()))
print('prediction differences:', int((with_memory != without_memory).sum().item()))
print('Caveat: qualitative memory diagnostic in simplified prototype.')

## 9. Save artifacts

In [ ]:
artifact_dir = Path('artifacts'); artifact_dir.mkdir(exist_ok=True, parents=True)
model_path = artifact_dir / 'titans_miras_promoter_classifier.pt'
config_path = artifact_dir / 'titans_miras_config.json'
torch.save(model.state_dict(), model_path)
config_path.write_text(json.dumps(config.to_dict(), indent=2))
print('saved', model_path)
print('saved', config_path)

## 10. Next steps

- Scale memory slots/depth and sequence length for longer contexts.
- Replace simplified EMA update with closer paper equations/objectives.
- Explore chunked streaming memory updates and richer retrieval objectives.